In [31]:
import kagglehub

path = kagglehub.dataset_download("farahfirdausa/flood-prediction-dataset")
print("Path to dataset files:", path)

Path to dataset files: C:\Users\raiti\.cache\kagglehub\datasets\farahfirdausa\flood-prediction-dataset\versions\1


In [32]:
import os
print(os.listdir(path))

['modis_flood_features_paling cleaning (1).csv']


In [33]:
import pandas as pd

df = pd.read_csv(f"{path}/modis_flood_features_paling cleaning (1).csv")
print(df.head())

         date         lon       lat  flooded  jrc_perm_water  precip_1d  \
0  2003-12-10  120.478677 -6.495942      1.0             1.0   9.645125   
1  2003-12-10  120.480923 -6.495942      1.0             1.0   9.645125   
2  2003-12-10  120.478677 -6.493697      0.0             1.0   9.645125   
3  2003-12-10  120.480923 -6.493697      1.0             0.0   9.645125   
4  2003-12-10  120.478677 -6.491451      0.0             1.0   9.645125   

   precip_3d        NDVI      NDWI  landcover  elevation     slope  \
0  28.935376  6190.93062 -0.051446       17.0        0.0  0.116685   
1  28.935376  6190.93062 -0.051446       17.0        1.0  0.839713   
2  28.935376  6190.93062  0.028399       17.0        0.0  0.466730   
3  28.935376  6190.93062  0.028399       17.0        4.0  0.939179   
4  28.935376  6190.93062 -0.002506       17.0        0.0  0.583403   

       aspect  upstream_area       TWI  target  
0  270.000000       0.031990  1.042098       0  
1  236.480074       0.031990  

In [34]:
print(df.shape)
print(df.isnull().sum())
print(df.dtypes)
print(df['target'].value_counts())

(1025801, 16)
date              0
lon               0
lat               0
flooded           0
jrc_perm_water    0
precip_1d         0
precip_3d         0
NDVI              0
NDWI              0
landcover         0
elevation         0
slope             0
aspect            0
upstream_area     0
TWI               0
target            0
dtype: int64
date               object
lon               float64
lat               float64
flooded           float64
jrc_perm_water    float64
precip_1d         float64
precip_3d         float64
NDVI              float64
NDWI              float64
landcover         float64
elevation         float64
slope             float64
aspect            float64
upstream_area     float64
TWI               float64
target              int64
dtype: object
target
0    979446
1     46355
Name: count, dtype: int64


In [35]:
# model will just predict "no flood" for almost everything and still get 95% accuracy, which is completely useless for a safety app
# undersample the majority class because we need all of minority class as it predicts flood

flood = df[df['target'] == 1]
no_flood = df[df['target'] == 0].sample(n=len(flood), random_state=42)

df_balanced = pd.concat([flood, no_flood]).sample(frac=1, random_state=42)
print(df_balanced['target'].value_counts())

target
1    46355
0    46355
Name: count, dtype: int64


In [36]:
print(df_balanced[['flooded', 'target']].value_counts())

flooded  target
1.0      1         46355
0.0      0         43062
1.0      0          3293
Name: count, dtype: int64


In [37]:
df_balanced["month"] = pd.to_datetime(df_balanced["date"]).dt.month
df_balanced[['month']].value_counts()

month
12       23530
6        22211
5        14813
7        11920
2        10628
1         9608
Name: count, dtype: int64

In [51]:
X = df_balanced[['lat', 'lon', 'month', 'elevation', 'precip_1d', 'precip_3d']]
y = df_balanced[['target']]

In [52]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [53]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,      # number of trees
    max_depth=6,           # how deep each tree grows
    learning_rate=0.1,     # how fast it learns
    random_state=42
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [54]:
y_pred = model.predict(X_test)

In [55]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.88      0.92      9340
           1       0.89      0.96      0.92      9202

    accuracy                           0.92     18542
   macro avg       0.92      0.92      0.92     18542
weighted avg       0.92      0.92      0.92     18542

[[8207 1133]
 [ 381 8821]]


In [60]:
import joblib
joblib.dump(model, "flood_model.pkl")

['flood_model.pkl']

In [57]:
print(df_balanced['target'].value_counts())
print(y_train.value_counts())

target
1    46355
0    46355
Name: count, dtype: int64
target
1         37153
0         37015
Name: count, dtype: int64


In [59]:
print(model.predict_proba([[-3.99, 120.01, 12, 5.0, 10.76, 75.37]]))

[[0.02569127 0.9743087 ]]


In [61]:
print(model.classes_)

[0 1]
